In [16]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# ==========================================
# 1. 禁用自动日志，全手动控制规避拦截
# ==========================================
try:
    mlflow.sklearn.autolog(disable=True)
    mlflow.autolog(disable=True)
except Exception:
    pass

# ==========================================
# 2. 准备纯 NumPy 数据（避免 Pandas 类型冲突）
# ==========================================
X, y = make_classification(n_samples=200, n_features=20, n_informative=5, n_redundant=2, random_state=42)
feature_names = [f"feature_{i}" for i in range(20)]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 3. 配置 AIDP 实验上下文
# ==========================================
experiment_name = "AIDP_Sklearn_Feature_Selection"
try:
    mlflow.set_experiment(experiment_name)
except Exception as e:
    print(f"设置实验提示: {e}")

# ==========================================
# 4. 模型构建与训练 
# ==========================================
print("正在训练特征选择 Pipeline 模型...")
with mlflow.start_run() as run:
    
    selector = SelectKBest(score_func=f_classif, k=5)
    classifier = RandomForestClassifier(n_estimators=50, random_state=42)
    
    pipeline = Pipeline([
        ('feature_selection', selector),
        ('classifier', classifier)
    ])
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"测试集准确率: {accuracy:.4f}")
    
    # 提取特征
    support_mask = pipeline.named_steps['feature_selection'].get_support()
    selected_features = [feature_names[i] for i, selected in enumerate(support_mask) if selected]
    print(f"成功筛选出的特征: {selected_features}")
    
    # ==========================================
    # 5. 手动向 MLflow 提报数据与【安全保存模型】
    # ==========================================
    mlflow.log_param("feature_selector", "SelectKBest")
    mlflow.log_param("selected_k", 5)
    mlflow.log_param("n_estimators", 50)
    mlflow.log_metric("accuracy", accuracy)
    
    # 保存特征为文本工件
    with open("selected_features.txt", "w") as f:
        f.write("\n".join(selected_features))
    mlflow.log_artifact("selected_features.txt", artifact_path="feature_analysis")
    
    # 【核心修复】：为 log_model 添加安全白名单，显式告知信任 f_classif 函数
    mlflow.sklearn.log_model(
        sk_model=pipeline, 
        artifact_path="feature_selection_model",
        skops_trusted_types=["sklearn.feature_selection._univariate_selection.f_classif"]
    )
    
    # 如果上面添加白名单依然受到环境拦截，你也可以直接采用下面的传统 pickle 备用方案：
    # mlflow.sklearn.log_model(
    #     sk_model=pipeline, 
    #     artifact_path="feature_selection_model",
    #     serialization_format="pickle"
    # )
    
    print(f"\n[成功] 模型已完全通过安全审计成功上传！Run ID: {run.info.run_id}")


正在训练特征选择 Pipeline 模型...


测试集准确率: 0.8500
成功筛选出的特征: ["feature_0", "feature_3", "feature_9", "feature_15", "feature_17"]



[成功] 模型已完全通过安全审计成功上传！Run ID: eaa0479c7b8e4796975a0dfe072f52e7


🏃 View run chill-goose-616 at: https://c1.df.aidp.us-ashburn-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.iad.amaaaaaaehf4braa7uebyl3nodprjmn5lgruzwqltl6j5diqnvu62qpkizzq/20240831/dataLakes/ocid1.aidataplatform.oc1.iad.amaaaaaaehf4braa7uebyl3nodprjmn5lgruzwqltl6j5diqnvu62qpkizzq/workspaces/8a0b87c4-029a-4082-90d8-66f516826ca1/mlops/#/experiments/21/runs/eaa0479c7b8e4796975a0dfe072f52e7
🧪 View experiment at: https://c1.df.aidp.us-ashburn-1.oci.oracleiaas.com/dataLakes/ocid1.aidataplatform.oc1.iad.amaaaaaaehf4braa7uebyl3nodprjmn5lgruzwqltl6j5diqnvu62qpkizzq/20240831/dataLakes/ocid1.aidataplatform.oc1.iad.amaaaaaaehf4braa7uebyl3nodprjmn5lgruzwqltl6j5diqnvu62qpkizzq/workspaces/8a0b87c4-029a-4082-90d8-66f516826ca1/mlops/#/experiments/21
